this workbook s the anaylsis for falls data i the connected bradford cohort. 

In [ ]:
library(ggplot2)
library(bigrquery)
library(tidyverse)
library(ggplot2)
library(lubridate)
library(stats)
library(finalfit)
ProjectId = "yhcr-prd-bradfor-bia-core"

targetdb3 <- 'yhcr-prd-bradfor-bia-core.CB_2172'
targetdb3 <- gsub(' ','',targetdb3)


In [ ]:
sql_fall <- paste(#combined bradford and airedale queries for injurious falls, searched for icd 10 codes for falls and fragility fractures in the first three recorded diagnosis becuase based on review on the data this was the most likely 
'
select *EXCEPT(ADMISSION_SEQUENCE)
from ',targetdb3,'.cb_2172_injurious_falls_v3
;',sep="")

tbl_fall <-bq_project_query(ProjectId,sql_fall)
ch_fall<- bq_table_download(tbl_fall)
ch_fall$person_id <- as.character(ch_fall$person_id)
print(ch_fall)


In [ ]:
sql1 <- 
paste(
    'SELECT person_id,dod,first_episodestartdate 
from ',targetdb3,'.care_home_cohort_v1 a 
order by person_id
;',sep="")

bp_tbl <-bq_project_query(ProjectId,sql1)
ch_cohort <- bq_table_download(bp_tbl)

ch_cohort$person_id <- as.character(ch_cohort$person_id)
print(ch_cohort)

In [ ]:

ch_fall_merged <- merge(x =ch_cohort, y = ch_fall, all.x = TRUE)

In [ ]:
ch_fall_merged$first_episodestartdate <- as.Date(ch_fall_merged$first_episodestartdate)

In [ ]:
ch_fall_merged$fall_date <- as.Date(ch_fall_merged$tbl_episode_start_date)
ch_fall_merged <- ch_fall_merged %>% select(-tbl_episode_start_date)

In [ ]:
ch_fall_merged <- ch_fall_merged %>% mutate(exposure_time= (dod-first_episodestartdate))

In [ ]:
ch_fall_merged$exposure_time <- as.numeric(ch_fall_merged$exposure_time)

In [ ]:
#for those with missing death dates, it waa assumed they had 12 months of exposure

In [ ]:
ch_fall_merged$exposure_time <- ifelse(is.na(ch_fall_merged$exposure_time),365,ch_fall_merged$exposure_time)

In [ ]:
#set the exposure time for those alive more 12 months afterwards to 365 days to calculate incidence density in the year after admission

In [ ]:
ch_fall_merged$exposure_time<- ifelse(ch_fall_merged$exposure_time > 365,365,ch_fall_merged$exposure_time)

In [ ]:
ch_fall_merged <- ch_fall_merged %>% mutate(fall = if_else(!is.na(fall_date),TRUE,FALSE))

Add hypertension diagnosis 

In [ ]:
sql_htn <- paste(#combined bradford and airedale queries for injurious falls 
'
select
*
from 
',targetdb3,'.cb_2172_hypertension_combined
where person_id in(select person_id from ',targetdb3,'.care_home_cohort_v1)
;',sep="")

tbl_htn <-bq_project_query(ProjectId,sql_htn)
ch_htn<- bq_table_download(tbl_htn)
ch_htn$person_id <- as.character(ch_htn$person_id)
print(ch_htn)

In [ ]:
ch_fall_merged <- merge(x=ch_fall_merged,y=ch_htn,by.x='person_id',by.y='person_id',all.x= TRUE)%>% select(-first_episodestartdate.y)

then we caluclate the inciddence rate by dividing the total number of falls in the cohort by the exposure time in person years

In [ ]:
(sum(ch_fall_merged$fall ==TRUE)/(ch_fall_merged$exposure_time %>% sum()/365))*100

the incidence rate for injurious falls in care home residents with hypertension rate was 10 falls per 100 person-years. 

In [ ]:
ch_fall_merged_htn <- ch_fall_merged %>% filter(hypertension == TRUE)

In [ ]:
table(ch_fall_merged_htn$fall)

In [ ]:
(sum(ch_fall_merged_htn$fall ==TRUE)/(ch_fall_merged_htn$exposure_time %>% sum()/365))*100